# Nigeria Multidimensional Poverty Index (MPI) Data Analysis

This notebook prepares household-level MPI data, examines deprivation patterns by LGA, produces geospatial visualizations, explores correlations, and tests machine-learning approaches for poverty classification.

> **Data note:** The source survey data and administrative boundaries are not included in this repository. Place authorized files in `../data/raw/` before running the notebook.

## 1. Setup and input paths

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

RAW_DIR = Path("../data/raw")
PROCESSED_DIR = Path("../data/processed")
OUTPUT_DIR = Path("../outputs")

INPUT_FILE = RAW_DIR / "nig21_National_office submit1.xlsx"
SHAPEFILE = RAW_DIR / "New_Shape_File" / "nga_admbnda_adm2_osgof_20170222.shp"

RANDOM_STATE = 42
MPI_THRESHOLD = 0.26

PROCESSED_DIR.mkdir(parents=True, exist_ok=True)
(OUTPUT_DIR / "figures").mkdir(parents=True, exist_ok=True)
(OUTPUT_DIR / "tables").mkdir(parents=True, exist_ok=True)

## 2. Load and prepare household-level data

In [ ]:
data = pd.read_excel(INPUT_FILE)
print(f"Rows: {len(data):,}")
print(f"Columns: {len(data.columns):,}")
data.head()

In [ ]:
# Keep one record per household while retaining household-level variables.
household_data = data.drop_duplicates(subset="Household ID").copy()

household_size = (
    data.groupby("Household ID")["Individual ID"]
    .count()
    .rename("HouseholdSize")
    .reset_index()
)

household_data = household_data.merge(household_size, on="Household ID", how="left")
household_data.reset_index(drop=True, inplace=True)

household_data.to_csv(PROCESSED_DIR / "household_level_data_with_size.csv", index=False)
print(f"Households: {len(household_data):,}")
household_data.head()

In [ ]:
Df = household_data.copy()
Df.info()

## 3. LGA coverage and aggregation

In [ ]:
unique_lgas = Df["LGA"].dropna().unique()
print("Total number of unique LGAs:", len(unique_lgas))

lga_counts = Df["LGA"].value_counts()
print("LGAs with more than 15 households:", (lga_counts > 15).sum())
lga_counts.head(10)

In [ ]:
indicator_columns = [
    "Nutrition", "Food security", "Health", "School attendance",
    "Years of schooling", "School lag", "Water access",
    "Water reliability", "Sanitation", "Housing", "Cooking",
    "Asset", "Unemployment", "Underemployment", "Security shock"
]

# Aggregate indicator counts and convert them to household percentages.
groupby_columns = ["LGA", "Zone", "State"]
Lga_df = Df.groupby(groupby_columns).sum(numeric_only=True)
total_households = Df.groupby(groupby_columns).size()

for col in indicator_columns:
    if col in Lga_df.columns:
        Lga_df[f"total_{col}"] = (Lga_df[col] / total_households) * 100

Lga_df = Lga_df.reset_index()
Lga_df.to_excel(OUTPUT_DIR / "tables" / "Lga_df.xlsx", index=False)

Lga_df.head()

## 4. Geospatial analysis

In [ ]:
import geopandas as gpd

shp_gdf = gpd.read_file(SHAPEFILE)
shp_gdf["admin2Name"] = shp_gdf["admin2Name"].str.upper()
shp_gdf = shp_gdf.rename(columns={"admin2Name": "LGA"})

merged_data = shp_gdf.merge(Lga_df, on="LGA", how="inner")
print(f"Mapped LGAs: {len(merged_data):,}")
merged_data.head()

In [ ]:
# Remove administrative fields that are not required for the analytical maps.
columns_to_drop = [
    "admin2Pcod", "admin2RefN", "admin2AltN", "admin2Al_1",
    "admin1Name", "admin1Pcod", "admin0Name", "admin0Pcod",
    "date", "validOn", "ValidTo"
]
merged_data = merged_data.drop(columns=[c for c in columns_to_drop if c in merged_data.columns])
merged_data.info()

### Selected LGA indicator maps

In [ ]:
def plot_lga_indicator(gdf, column, title):
    fig, ax = plt.subplots(figsize=(15, 10))
    gdf.plot(ax=ax, color="lightgray", edgecolor="white", linewidth=0.2)
    gdf.plot(column=column, linewidth=0.5, edgecolor="black", legend=True, ax=ax)
    ax.set_title(title)
    ax.set_axis_off()
    plt.tight_layout()
    plt.show()

for column in ["total_Sanitation", "total_Food security", "total_School attendance"]:
    if column in merged_data.columns:
        plot_lga_indicator(merged_data, column, column.replace("total_", ""))

## 5. Correlation analysis

In [ ]:
# Keep numeric deprivation indicators and exclude identifiers, weights and geographic fields.
metadata_columns = [
    "Zone", "State", "LGA", "HH No", "Household_ID", "Age", "Sex",
    "Counting Vector", "HouseholdSize", "MPI Identification with k=26%",
    "Individual Weight", "Household Weight", "Population Weight"
]

corr_df = Df.drop(columns=[c for c in metadata_columns if c in Df.columns], errors="ignore").copy()
corr_df = corr_df.select_dtypes(include=np.number)
corr_matrix = corr_df.corr()

plt.figure(figsize=(18, 12))
sns.heatmap(corr_matrix, annot=True, fmt=".2f", linewidths=0.5)
plt.title("Correlation Matrix of Numeric MPI Indicators")
plt.tight_layout()
plt.show()

## 6. Machine-learning classification

In [ ]:
# Construct the binary target from the 26% MPI threshold.
if "MPI Identification with k=26%" not in Df.columns:
    raise KeyError("The dataset must contain 'MPI Identification with k=26%'.")

ml_df = Df.copy()
ml_df["Poverty"] = (ml_df["MPI Identification with k=26%"] <= MPI_THRESHOLD).astype(int)

exclude_columns = [
    "MPI Identification with k=26%", "Poverty", "Zone", "State", "LGA",
    "HH No", "Household_ID", "Age", "Sex", "Counting Vector",
    "HouseholdSize", "Individual Weight", "Household Weight", "Population Weight"
]

X = ml_df.drop(columns=[c for c in exclude_columns if c in ml_df.columns], errors="ignore")
X = X.select_dtypes(include=np.number)
y = ml_df["Poverty"]

print("Feature matrix:", X.shape)
print("Class distribution:")
print(y.value_counts(normalize=False))

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=RANDOM_STATE, stratify=y
)

models = {
    "Random Forest": RandomForestClassifier(
        n_estimators=100, class_weight="balanced", random_state=RANDOM_STATE
    ),
    "Logistic Regression": Pipeline([
        ("scaler", StandardScaler()),
        ("model", LogisticRegression(max_iter=2000, random_state=RANDOM_STATE))
    ]),
    "SVM": Pipeline([
        ("scaler", StandardScaler()),
        ("model", SVC(kernel="rbf", random_state=RANDOM_STATE))
    ]),
    "Gradient Boosting": GradientBoostingClassifier(
        n_estimators=100, random_state=RANDOM_STATE
    )
}

results = []
for name, model in models.items():
    model.fit(X_train, y_train)
    predictions = model.predict(X_test)
    results.append({"Model": name, "Accuracy": accuracy_score(y_test, predictions)})
    print(f"\n{name}")
    print(classification_report(y_test, predictions, zero_division=0))
    print("Confusion matrix:")
    print(confusion_matrix(y_test, predictions))

model_results = pd.DataFrame(results).sort_values("Accuracy", ascending=False)
model_results

### Random Forest feature importance

In [ ]:
rf_model = models["Random Forest"]
feature_importance = (
    pd.Series(rf_model.feature_importances_, index=X.columns, name="Importance")
    .sort_values(ascending=False)
)

plt.figure(figsize=(10, 7))
feature_importance.head(20).sort_values().plot(kind="barh")
plt.xlabel("Feature importance")
plt.title("Random Forest Feature Importance")
plt.tight_layout()
plt.show()

feature_importance.to_csv(OUTPUT_DIR / "tables" / "random_forest_feature_importance.csv")

### Cross-validation

In [ ]:
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
cv_model = RandomForestClassifier(n_estimators=100, random_state=RANDOM_STATE)
cv_scores = cross_val_score(cv_model, X, y, scoring="accuracy", cv=cv)

print("Fold accuracy scores:", cv_scores)
print(f"Mean accuracy: {cv_scores.mean():.3f}")
print(f"Standard deviation: {cv_scores.std():.3f}")

## 7. Interpretation and limitations

The machine-learning experiments are exploratory. The binary target is derived from the MPI identification measure, while the predictors include the underlying deprivation indicators. High predictive performance therefore does not establish causality or demonstrate that the model is an independent measure of poverty.

LGA-level percentages should also be interpreted alongside the number of observations available for each geographic unit and the survey design and weighting procedures used to produce official estimates.